# Exportar el modelo final
Reentrena **la misma receta ganadora** del proyecto (variante 4b: sobremuestreo con imágenes reales + pesos de clase + ajuste fino) y guarda el modelo para usarlo en la web. Tarda unos 8–10 minutos con GPU T4.

> Activar GPU: Runtime → Change runtime type → T4 GPU.

In [ ]:
import os, sys
REPO = "/content/ia-Kaggle-Coffee"
if os.path.exists("/content") and not os.path.isdir(REPO):
    !git clone -q -b claude/brave-archimedes-jnuxk1 https://github.com/JavierAnd0/ia-Kaggle-Coffee {REPO}
if os.path.isdir(REPO):
    %cd {REPO}
sys.path.insert(0, os.getcwd())
!pip -q install keras-tuner
import pandas as pd, tensorflow as tf
from src import config, generators, tuning, export
from src import train as trainer
N_EXTRA = 150     # imágenes reales repetidas de la clase minoritaria (igual que en la comparación)
EPOCHS, FT_EPOCHS = 30, 10

In [ ]:
tr = pd.read_csv(config.SPLITS_DIR / "train.csv")
va = pd.read_csv(config.SPLITS_DIR / "val.csv")
te = pd.read_csv(config.SPLITS_DIR / "test.csv")
li = generators.make_label_index(tr["label"])
best = tuning.load_best_hp()
minority = tr["label"].value_counts().idxmin()
extra = tr[tr["label"] == minority].sample(N_EXTRA, replace=True, random_state=config.SEED)
tr_aug = pd.concat([tr, extra], ignore_index=True)
print(li, len(tr), '->', len(tr_aug))

In [ ]:
model = trainer.train_model(best, tr_aug, va, li, epochs=EPOCHS, use_class_weight=True,
                            finetune_epochs=FT_EPOCHS, weight_df=tr, verbose=2)

In [ ]:
from sklearn.metrics import classification_report
res = {}
for name, d in (("val", va), ("test", te)):
    ev = trainer.evaluate(model, best, d, li)
    res[name] = {k: round(v, 4) for k, v in ev.items() if k not in ("y_true", "y_pred")}
    print(name, res[name])
ev = trainer.evaluate(model, best, te, li)
print(classification_report(ev['y_true'], ev['y_pred'], target_names=sorted(li, key=li.get), zero_division=0))
print(trainer.confusion_df(ev, li))

In [ ]:
out = export.export_model(model, li, best, res, config.ROOT / "models")
!ls -la {out}
!zip -qr modelo_cafe.zip models
from google.colab import files
files.download("modelo_cafe.zip")   # descarga el zip; luego súbelo al repo en la carpeta models/